## Cell 1: Setup & Configuration

INTENT:
1. Mount Drive so we can access the uploaded data (if stored there).
2. Install fastparquet (for reading .parquet efficiently) and networkx (for graphs).
3. Import all required libraries.
4. Define a Config class/dict to hold paths and tunable hyperparameters.
5. Validate that the data folders exist.


In [ ]:
import os
import sys
import json
import gc
import time
from datetime import datetime
from collections import defaultdict, Counter
from typing import Dict, List, Tuple, Optional, Union, Any
from zipfile import ZipFile

# Data manipulation
import numpy as np
import pandas as pd

# For efficient parquet reading (int8 embeddings)
import pyarrow.parquet as pq

# For sparse matrices and community detection
from scipy.sparse import csr_matrix, coo_matrix
from scipy.stats import percentileofscore

# For graph-based candidate generation (Louvain community detection)
import networkx as nx
from networkx.algorithms.community import louvain_communities

# For approximate nearest neighbor (to handle scale, though we start brute-force)
from sklearn.neighbors import NearestNeighbors

# For progress bars and warnings
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# For optional Google Drive mounting
from google.colab import drive

print("All core imports successful.")

In [ ]:
import zipfile
import os

# Define the paths for the zip files and their extraction destinations
dev_zip_path = "/content/dev-20260703T125103Z-3-001.zip"
eval_zip_path = "/content/eval-20260703T125054Z-3-001.zip"

dev_extract_path = "/content/data/"
eval_extract_path = "/content/data/"

# Create target directories if they don't exist
os.makedirs(dev_extract_path, exist_ok=True)
os.makedirs(eval_extract_path, exist_ok=True)

print(f"Extracting {os.path.basename(dev_zip_path)} to {dev_extract_path}")
with zipfile.ZipFile(dev_zip_path, 'r') as zip_ref:
    zip_ref.extractall(dev_extract_path)
print(f"Successfully extracted {os.path.basename(dev_zip_path)}.")

print(f"Extracting {os.path.basename(eval_zip_path)} to {eval_extract_path}")
with zipfile.ZipFile(eval_zip_path, 'r') as zip_ref:
    zip_ref.extractall(eval_extract_path)
print(f"Successfully extracted {os.path.basename(eval_zip_path)}.")

print("Data extraction complete.")

In [ ]:
# Path and Hyperparameter Configuration
#these are highly adjustable parameters that will greatly influence the outputs
class Config:
    """
    Central configuration object for the entire pipeline.
    All tunable parameters are defined here to avoid hard-coding magic numbers.
    """
    DEV_PATH = "/content/data/dev/"
    EVAL_PATH = "/content/data/eval/"

    # Temporal parameters
    # The maximum time window (in seconds) within which we consider posts
    # potentially coordinated. 30 minutes = 1800 seconds.
    MAX_TIME_WINDOW_SEC = 1800

    # The number of seconds to bucket timestamps for burst detection.
    # 5 minutes = 300 seconds.
    BUCKET_SEC = 300

    # Embedding parameters
    # The int8 embeddings need to be scaled to float32 to compute cosine.
    # Typically, int8 quantization in sentence-transformers uses a scale of 1/127.
    EMBEDDING_SCALE = 1.0 / 127.0

    # For Sniffer C (semantic), we only consider pairs with cosine similarity
    # above this threshold. We set it relatively high to avoid linking generic news.
    SEMANTIC_SIM_THRESHOLD = 0.85

    # Candidate generation parameters
    # Minimum number of posts a candidate cluster must have.
    MIN_CLUSTER_SIZE = 3

    # Null model calibration
    # Number of permutations for the null distribution.
    NULL_PERMUTATIONS = 5000

    # Percentile threshold for flagging (99.9% = very strict).
    # reduce this for less strictness in flagging noise vs signal
    PERCENTILE_THRESHOLD = 97.5

    # Output
    # Name of the results file to produce.
    RESULTS_FILENAME = "results.json"

    # Debug / Verbosity
    VERBOSE = True

    @classmethod
    def get_data_path(cls, tier: str = "dev") -> str:
        """Return the appropriate base path for the given tier."""
        if tier == "dev":
            return cls.DEV_PATH
        elif tier == "eval":
            return cls.EVAL_PATH
        else:
            raise ValueError(f"Unknown tier: {tier}. Must be 'dev' or 'eval'.")

In [ ]:
#just a test of the config class
config = Config()

config.get_data_path(tier="eval")

In [ ]:
# Validate paths and optionally mount Drive

def validate_and_mount():
    """Check if the data paths exist. If not, try to mount Google Drive."""
    config = Config()

    # Check dev path first as a representative test
    dev_posts = os.path.join(config.DEV_PATH, "posts.jsonl")
    if not os.path.exists(dev_posts):
        print("Data not found in /content/data/.")
        print("Attempting to mount Google Drive...")
        drive.mount('/content/drive')

        # Update the paths in Config to point to your actual Drive location.
        # Since we don't know the exact path, we print a warning.
        print("\n" + "="*60)
        print("WARNING: Please update Config.DEV_PATH and Config.EVAL_PATH")
        print("to match the actual location of your data in Google Drive.")
        print("Example: /content/drive/MyDrive/path/to/dev/")
        print("="*60 + "\n")
        return False
    else:
        print(f"✅ Data found at: {config.DEV_PATH}")
        return True

if validate_and_mount():
    print("\n✅ Setup complete. Ready to proceed to Cell 2.")
else:
    print("\n⚠️ Please adjust the paths in Config and re-run this cell.")

My reasoning for this layout:

I put all tunable knobs (time windows, thresholds, null permutations) in a Config class. You can tweak these based on manual inspection without hunting through spaghetti code.

I explicitly import networkx and sklearn early because Colab doesn't always have them pre-installed (though it usually does). If we need to install, we can add a !pip install line at the top.

The path validation checks for dev/ first. If it fails, it assumes the data might be in Google Drive and attempts to mount it, leaving the final path update to you (since everyone's folder structure is slightly different).

## Cell 2: Data Loaders & Profile Prior Computation


Intent of this cell:
We need to load three different file formats (.jsonl, .jsonl, .parquet) efficiently and consistently. Instead of writing one monolithic function, I decompose the loading into three dedicated functions with clear contracts:

load_accounts() – reads accounts, extracts metadata, and computes a profile_prior score (higher = more suspicious).

load_posts() – reads posts, parses ISO timestamps to datetime objects, and extracts all interaction fields (mentions, thread_id, urls, etc.).

load_embeddings() – reads the 1024‑dimensional int8 embeddings, scales them to float32 (so cosine similarity works correctly), and returns a dictionary mapping post_id -> vector.

We also implement a consistency check at the end to ensure all post IDs in posts.jsonl have a corresponding vector in embeddings.parquet (and warn if some are missing, which would break downstream similarity calculations).

The compute_profile_prior() function is where we encode the "empty profile" and "low followers / high posts" heuristics we discovered during stress-testing.

In [ ]:
import json
import os
from datetime import datetime
from typing import Dict, List, Optional, Tuple
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from tqdm.notebook import tqdm

# 2.1: Account Loader & Profile Prior

def compute_profile_prior(profile_dict: dict, account_id: str) -> float:
    """
    Compute a "suspicion score" for a single account based on profile metadata.

    The score ranges from 0.0 (likely organic) to 1.0 (highly suspicious).
    We combine three signals:
      1. Empty profile (profile_dict is empty or missing critical fields).
      2. Extremely low follower count combined with high post count
         (post_to_follower_ratio > 20 is a red flag).
      3. Is the account verified? Verified accounts get a penalty (lower score).

    Args:
        profile_dict: The nested "profile" object from accounts.jsonl.
        account_id: The account ID (used only for debug logging).

    Returns:
        float: A prior probability (0.0–1.0) that this account is a coordinated actor.

    Example:
        profile = {"metadata": {"followers_count": 24, "posts_count": 4029}}
        >> compute_profile_prior(profile, "acct_123") -> 0.85
    """

    # Default: assume organic unless we see evidence
    score = 0.0

    # If the profile is completely empty -> massive red flag
    if not profile_dict:
        return 0.95

    # Extract metadata safely
    metadata = profile_dict.get("metadata", {})
    followers = metadata.get("followers_count", 0)
    posts = metadata.get("posts_count", 0)
    verified = metadata.get("verified", False)
    # Also check the top-level "favourites_count" if present
    favourites = profile_dict.get("favourites_count", 0)

    # Signal 1: Empty metadata
    # If metadata dict is empty but profile is not entirely empty (rare), we still penalise
    if not metadata:
        score += 0.4

    # Signal 2: Post-to-Follower Ratio
    # Organic users typically have posts_count <= 10 * followers_count.
    # Coordinated / bot accounts often have high post volume but low following.
    if followers > 0 and posts > 0:
        ratio = posts / followers
        if ratio > 50:
            score += 0.5   # Extremely suspicious
        elif ratio > 20:
            score += 0.3   # Moderately suspicious
        elif ratio > 10:
            score += 0.1   # Slightly suspicious
    elif followers == 0 and posts > 100:
        # 0 followers but many posts -> almost certainly a bot
        score += 0.7

    # Signal 3: Verified status
    # Blue-check accounts are less likely to be dedicated coordination burners
    if verified:
        score -= 0.2   # Give them the benefit of the doubt

    # Signal 4: Extremely high followers
    # Very high follower accounts (e.g., 44k) are usually organic influencers,
    # even if they post the same content. We penalise them slightly.
    if followers > 50000:
        score -= 0.3
    elif followers > 10000:
        score -= 0.1

    # Clamp the score to [0.0, 1.0]
    score = max(0.0, min(1.0, score))
    return score

In [ ]:
def load_accounts(filepath: str) -> Tuple[Dict[str, Dict], Dict[str, float]]:
    """
    Load accounts.jsonl and compute a profile_prior for each account.

    Args:
        filepath: Path to the accounts.jsonl file.

    Returns:
        A tuple (account_profiles, profile_priors) where:
          - account_profiles: dict[account_id] = full profile dict.
          - profile_priors: dict[account_id] = computed suspicion score (0.0–1.0).

    Raises:
        FileNotFoundError: If the file does not exist.
        json.JSONDecodeError: If a line is malformed.
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Accounts file not found: {filepath}")

    account_profiles = {}
    profile_priors = {}

    with open(filepath, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(tqdm(f, desc="Loading accounts"), start=1):
            line = line.strip()
            if not line:
                continue
            try:
                data = json.loads(line)
                account_id = data.get("account_id")
                profile = data.get("profile", {})

                if account_id is None:
                    print(f"Warning: line {line_num} missing account_id, skipping.")
                    continue

                # Store the raw profile
                account_profiles[account_id] = profile
                # Compute and store the prior
                profile_priors[account_id] = compute_profile_prior(profile, account_id)

            except json.JSONDecodeError as e:
                print(f"Warning: Could not parse line {line_num}: {e}")
                continue

    return account_profiles, profile_priors

In [ ]:
def parse_created_at(ts_str: str) -> datetime:
    """
    Parse an ISO-8601 timestamp string to datetime.
    Handles both "YYYY-MM-DDTHH:MM:SS" and variations.

    Args:
        ts_str: Timestamp string.

    Returns:
        datetime object.

    Raises:
        ValueError: If parsing fails.
    """
    # The dataset uses "2026-04-15T17:47:28" (no timezone).
    # We'll assume UTC.
    return datetime.fromisoformat(ts_str)

In [ ]:
def load_posts(filepath: str) -> Tuple[Dict[str, dict], List[dict]]:
    """
    Load posts.jsonl and perform basic parsing & validation.

    We return both a dictionary (fast lookup by post_id) and a list
    (convenient for iteration) to give flexibility in later phases.

    The 'created_at' field is converted to a datetime object for temporal analysis.

    Args:
        filepath: Path to the posts.jsonl file.

    Returns:
        A tuple (posts_dict, posts_list) where:
          - posts_dict: dict[post_id] = post record (with 'created_at' as datetime).
          - posts_list: list of all post records (same dicts).

    Raises:
        FileNotFoundError: If the file does not exist.
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Posts file not found: {filepath}")

    posts_dict = {}
    posts_list = []

    with open(filepath, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(tqdm(f, desc="Loading posts"), start=1):
            line = line.strip()
            if not line:
                continue
            try:
                post = json.loads(line)
                post_id = post.get("post_id")

                if post_id is None:
                    print(f"Warning: line {line_num} missing post_id, skipping.")
                    continue

                # Convert timestamp to datetime object
                if "created_at" in post:
                    post["created_at"] = parse_created_at(post["created_at"])
                else:
                    print(f"Warning: post {post_id} has no created_at, setting to None.")
                    post["created_at"] = None

                # Ensure interaction fields are lists (sometimes they may be missing)
                post.setdefault("mentions", [])
                post.setdefault("hashtags", [])
                post.setdefault("urls", [])
                post.setdefault("reply_to_post_id", None)
                post.setdefault("thread_id", None)
                post.setdefault("quoted_post_id", None)

                posts_dict[post_id] = post
                posts_list.append(post)

            except json.JSONDecodeError as e:
                print(f"Warning: Could not parse line {line_num}: {e}")
                continue
            except ValueError as e:
                print(f"Warning: Timestamp parsing error for line {line_num}: {e}")
                continue

    return posts_dict, posts_list

In [ ]:
# Embedding Loader
def load_embeddings(filepath: str) -> Dict[str, np.ndarray]:
    """
    Load embeddings.parquet and convert the int8 vectors to float32.

    The parquet file contains columns: 'post_id' and a binary/array column for vectors.
    According to the task, the embeddings are int8 quantised (not unit-normalised).
    We scale them by Config.EMBEDDING_SCALE (1/127) to bring them into [-1, 1]
    so that cosine similarity is meaningful.

    Args:
        filepath: Path to embeddings.parquet.

    Returns:
        dict[post_id] = np.ndarray of shape (1024,) dtype=float32.

    Raises:
        FileNotFoundError: If the file does not exist.
        ValueError: If the parquet file has an unexpected schema.
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Embeddings file not found: {filepath}")

    # Read the parquet file using pyarrow
    table = pq.read_table(filepath)

    # Convert to pandas for easier manipulation (11k rows is small)
    df = table.to_pandas()

    # The column containing the vectors may be named 'vector', 'embedding', or similar.
    # We need to sniff the column names.
    possible_vector_columns = ['vector', 'embedding', 'embeddings', 'vec']
    vector_col = None
    for col in possible_vector_columns:
        if col in df.columns:
            vector_col = col
            break

    if vector_col is None:
        raise ValueError(
            f"Could not identify vector column. Found columns: {df.columns.tolist()}"
        )

    # Ensure we have a post_id column
    if 'post_id' not in df.columns:
        raise ValueError("Embeddings file missing 'post_id' column.")

    embeddings_dict = {}
    scale = 1.0 / 127.0  # int8 -> float32 conversion

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Loading embeddings"):
        post_id = row['post_id']
        # The vector may be stored as a list or a numpy array
        vec = row[vector_col]
        if isinstance(vec, (np.ndarray, list)):
            vec = np.array(vec, dtype=np.float32) * scale
        else:
            # Sometimes it's stored as a bytes object (Arrow binary) -> convert
            try:
                vec = np.frombuffer(vec, dtype=np.int8).astype(np.float32) * scale
            except Exception as e:
                print(f"Warning: Could not parse vector for post {post_id}: {e}")
                continue

        # Safety check: ensure the vector is 1024-dimensional
        if len(vec) != 1024:
            print(f"Warning: post {post_id} has vector dimension {len(vec)}, expected 1024.")
            # If it's still workable, we keep it; otherwise skip.
            if len(vec) != 1024:
                continue

        embeddings_dict[post_id] = vec

    return embeddings_dict

In [ ]:
# Consistency Check & Summary Statistics

def print_data_summary(account_profiles: dict, posts_list: list, embeddings_dict: dict):
    """Print a human-readable summary of the loaded data."""
    print("\n" + "="*60)
    print("DATA LOADING SUMMARY")
    print("="*60)
    print(f"✅ Loaded {len(account_profiles)} accounts.")
    print(f"✅ Loaded {len(posts_list)} posts.")
    print(f"✅ Loaded {len(embeddings_dict)} embeddings.")

    # Check how many posts have embeddings
    posts_with_embeddings = sum(1 for p in posts_list if p['post_id'] in embeddings_dict)
    print(f"✅ {posts_with_embeddings} / {len(posts_list)} posts have corresponding embeddings.")

    missing = len(posts_list) - posts_with_embeddings
    if missing > 0:
        print(f"⚠️  WARNING: {missing} posts are missing embeddings! They will be dropped from semantic analysis.")

    # Compute average profile prior to see if data is generally suspicious
    if account_profiles:
        # We don't have priors here, they were returned by load_accounts
        pass

    # Show language distribution
    lang_counts = Counter()
    for post in posts_list:
        lang = post.get("language", "unknown")
        lang_counts[lang] += 1

    print("\n📊 Language distribution (top 10):")
    for lang, count in lang_counts.most_common(10):
        print(f"   {lang}: {count} posts")

In [ ]:
#Main Execution for data loading and parsing

# Which tier are we working on? We'll default to 'dev' for now.
TIER = "eval"  # Change to "eval" later for final results.

# Get the base data path
base_path = Config.get_data_path(TIER)
print(f"\n📂 Loading data from: {base_path}")

# Load accounts
account_profiles, profile_priors = load_accounts(
    os.path.join(base_path, "accounts.jsonl")
)
print(f"✅ Loaded {len(account_profiles)} accounts with computed profile priors.")

# Load posts
posts_dict, posts_list = load_posts(
    os.path.join(base_path, "posts.jsonl")
)
print(f"✅ Loaded {len(posts_list)} posts.")

# Load embeddings
embeddings_dict = load_embeddings(
    os.path.join(base_path, "embeddings.parquet")
)
print(f"✅ Loaded {len(embeddings_dict)} embedding vectors.")

# Print summary
print_data_summary(account_profiles, posts_list, embeddings_dict)



# For clarity, we'll use a dict to hold everything.
DATA = {
    "account_profiles": account_profiles,
    "profile_priors": profile_priors,
    "posts_dict": posts_dict,
    "posts_list": posts_list,
    "embeddings_dict": embeddings_dict,
    "tier": TIER,
}

In [ ]:
# funbction to view the data summary
# has a tunable argument num_items_to_view configurable to user preference
def view_data_summary(num_items_to_view: int = 5):
    """
    Displays a summary of the DATA dictionary, showing a tunable number of items
    for each top-level key.

    Args:
        num_items_to_view: The maximum number of items to display for each entry.
    """
    print(f"\n--- Viewing up to {num_items_to_view} items from DATA dictionary ---")
    for key, value in DATA.items():
        print(f"\nKey: {key}")
        print(f"  Type: {type(value)}")
        if isinstance(value, (dict, list, tuple, set)):
            print(f"  Total items: {len(value)}")
            if isinstance(value, dict):
                print("  Sample items:")
                for i, (k, v) in enumerate(value.items()):
                    if i >= num_items_to_view:
                        break
                    # Print key and type/length of value
                    if isinstance(v, (dict, list, tuple, set)):
                        print(f"    - {k}: {type(v).__name__} (len={len(v)})")
                    elif isinstance(v, np.ndarray):
                        print(f"    - {k}: np.ndarray (shape={v.shape}, dtype={v.dtype})")
                    else:
                        print(f"    - {k}: {type(v).__name__}")
            elif isinstance(value, (list, tuple)):
                print("  Sample items:")
                for i, item in enumerate(value):
                    if i >= num_items_to_view:
                        break
                    if isinstance(item, (dict, list, tuple, set)):
                        print(f"    - {type(item).__name__} (len={len(item)})")
                    elif isinstance(item, np.ndarray):
                        print(f"    - np.ndarray (shape={item.shape}, dtype={item.dtype})")
                    else:
                        print(f"    - {type(item).__name__}")
            else:
                print(f"  Value: {str(value)[:50]}{'...' if len(str(value)) > 50 else ''}")
        else:
            print(f"  Value: {str(value)[:50]}{'...' if len(str(value)) > 50 else ''}")

view_data_summary(num_items_to_view=10) # You can change 3 to any number

## Cell 3: The 3 Sniffers & Hypergraph Assembly

INTENT:
1. Build temporal buckets (e.g., 30-minute windows) to limit search space
   and enforce temporal locality.
2. Implement three independent "sniffer" functions, each producing a list
of weighted edges (account_i, account_j, weight) based on a specific signal.
3. Merge all edges into a single NetworkX graph.
4. Run Louvain community detection to extract dense subgraphs (candidate clusters).
5. For each candidate cluster, compute an initial aggregate score (to be refined
in Cell 4).
6. Return candidate_clusters with post_ids, account_ids, and raw scores.


In [ ]:
import numpy as np
import networkx as nx
from collections import defaultdict, Counter
from datetime import datetime, timedelta
from sklearn.metrics.pairwise import cosine_similarity
from networkx.algorithms.community import louvain_communities
from tqdm.notebook import tqdm
import math
from typing import List, Tuple, Dict, Set, Optional

# Temporal Bucketing

def build_temporal_buckets(posts_list: List[dict], bucket_sec: int) -> Dict[int, List[str]]:
    """
    Group post_ids by time buckets (e.g., 30-minute intervals).

    This is used to limit the pairwise search for Sniffer C (semantic).
    Two posts that are far apart in time (> bucket_sec) will never be linked,
    even if they are semantically identical. This enforces temporal locality
    and drastically reduces the O(n²) cost.

    Args:
        posts_list: List of post dicts (must have 'post_id' and 'created_at').
        bucket_sec: Width of each bucket in seconds (e.g., 1800 for 30 min).

    Returns:
        dict[bucket_start_timestamp] = list of post_ids in that bucket.
        Bucket start is the epoch second of the bucket's beginning.
    """
    buckets = defaultdict(list)
    for post in posts_list:
        post_id = post['post_id']
        dt = post['created_at']
        if dt is None:
            continue
        # Compute bucket start: floor the datetime to the bucket_sec
        bucket_start = int(dt.timestamp() // bucket_sec) * bucket_sec
        buckets[bucket_start].append(post_id)
    return dict(buckets)

In [ ]:
#Sniffer A – Structural Overlap

def sniffer_structure(posts_dict: Dict[str, dict]) -> List[Tuple[str, str, float]]:
    """
    Sniffer A: Find accounts that reply to the exact same parent post,
    share the same thread_id, or quote the same post.

    This catches the "same digital room" behaviour we saw with the two French
    accounts replying to p_135860475104035.

    Strategy:
        - Group post_ids by (thread_id, reply_to_post_id, quoted_post_id)
        - For each group, if there are at least 2 distinct accounts, add edges.
        - Weight = 0.8 for exact reply_to, 0.6 for thread_id, 0.5 for quoted.
          (We give highest weight to direct replies because they are the most
           deliberate signal.)

    Args:
        posts_dict: dict[post_id] -> post record.

    Returns:
        List of (account_i, account_j, weight) tuples.
    """
    edges = []

    # We'll collect three separate groupings
    # 1. Direct replies to the same parent (reply_to_post_id)
    reply_groups = defaultdict(list)
    # 2. Same thread_id (conversation tree)
    thread_groups = defaultdict(list)
    # 3. Quoting the same post
    quote_groups = defaultdict(list)

    for post_id, post in posts_dict.items():
        account_id = post['account_id']

        # Direct reply
        reply_to = post.get('reply_to_post_id')
        if reply_to is not None:
            reply_groups[reply_to].append((account_id, post_id))

        # Thread
        thread_id = post.get('thread_id')
        if thread_id is not None:
            thread_groups[thread_id].append((account_id, post_id))

        # Quote
        quoted = post.get('quoted_post_id')
        if quoted is not None:
            quote_groups[quoted].append((account_id, post_id))

    # Helper function to add edges from a grouping
    def add_edges_from_group(group_dict, weight, min_accounts=2):
        for target, members in group_dict.items():
            if len(members) < min_accounts:
                continue
            # Get unique accounts in this group
            accounts = set(acc for acc, _ in members)
            if len(accounts) < min_accounts:
                continue
            # Connect every pair of distinct accounts in this group
            accounts_list = list(accounts)
            for i in range(len(accounts_list)):
                for j in range(i+1, len(accounts_list)):
                    edges.append((accounts_list[i], accounts_list[j], weight))

    add_edges_from_group(reply_groups, weight=0.8)
    add_edges_from_group(thread_groups, weight=0.6)
    add_edges_from_group(quote_groups, weight=0.5)

    return edges

In [ ]:
#Sniffer B – Entity Overlap (Mentions, URLs, Hashtags)
def compute_idf(posts_list: List[dict], field: str) -> Dict[str, float]:
    """
    Compute Inverse Document Frequency (IDF) for a given field (mentions, urls, hashtags).

    Rare entities get higher IDF, so they contribute more weight to an edge.
    This prevents common tokens like "#UNRWA" from drowning out the signal.

    Args:
        posts_list: List of post dicts.
        field: One of 'mentions', 'urls', 'hashtags'.

    Returns:
        dict[entity] -> IDF weight.
    """
    doc_freq = Counter()
    total_docs = len(posts_list)

    for post in posts_list:
        entities = post.get(field, [])
        if not entities:
            continue
        # Deduplicate entities within a single post (so a post doesn't count twice)
        unique_entities = set(entities)
        for entity in unique_entities:
            doc_freq[entity] += 1

    # Compute IDF: log((N+1) / (df+1)) + 1 to smooth
    idf = {}
    for entity, df in doc_freq.items():
        idf[entity] = math.log((total_docs + 1) / (df + 1)) + 1.0
    return idf

In [ ]:
def sniffer_entity(posts_dict: Dict[str, dict], posts_list: List[dict]) -> List[Tuple[str, str, float]]:
    """
    Sniffer B: Find accounts that share the same rare mentions, URLs, or hashtags.

    Strategy:
        1. Compute IDF weights for mentions, URLs, and hashtags across all posts.
        2. For each entity (mention/url/hashtag), get all accounts that used it.
        3. Connect every pair of accounts that share the same entity,
           with weight = entity's IDF * 0.4 (to keep it on par with structural edges).

    Args:
        posts_dict: dict[post_id] -> post record.
        posts_list: list of all posts (needed for IDF computation).

    Returns:
        List of (account_i, account_j, weight) tuples.
    """
    # Compute IDF for each entity type
    print("Computing IDF for mentions...")
    idf_mentions = compute_idf(posts_list, 'mentions')
    print("Computing IDF for URLs...")
    idf_urls = compute_idf(posts_list, 'urls')
    print("Computing IDF for hashtags...")
    idf_hashtags = compute_idf(posts_list, 'hashtags')

    # Build maps: entity -> set of (account_id, post_id)
    entity_to_accounts = defaultdict(set)

    for post_id, post in posts_dict.items():
        account_id = post['account_id']
        # Mentions
        for mention in post.get('mentions', []):
            if mention in idf_mentions:
                entity_to_accounts[('mention', mention)].add(account_id)
        # URLs
        for url in post.get('urls', []):
            # We can use the full URL or domain. Using full URL is more specific.
            if url in idf_urls:
                entity_to_accounts[('url', url)].add(account_id)
        # Hashtags
        for hashtag in post.get('hashtags', []):
            if hashtag in idf_hashtags:
                entity_to_accounts[('hashtag', hashtag)].add(account_id)

    # Now generate edges
    edges = []
    # We add edges only if at least 2 distinct accounts share that entity
    for (entity_type, entity), accounts in entity_to_accounts.items():
        if len(accounts) < 2:
            continue
        # Get the IDF weight for this entity
        if entity_type == 'mention':
            weight = idf_mentions.get(entity, 0.0)
        elif entity_type == 'url':
            weight = idf_urls.get(entity, 0.0)
        elif entity_type == 'hashtag':
            weight = idf_hashtags.get(entity, 0.0)
        else:
            weight = 0.0

        # Scale the weight: we want entity edges to be meaningful but not overpower
        # structural edges (which are 0.5-0.8). So we multiply by 0.4 and cap at 0.7.
        weight = min(0.7, weight * 0.4)
        if weight < 0.05:
            continue  # ignore extremely rare or common entities

        accounts_list = list(accounts)
        for i in range(len(accounts_list)):
            for j in range(i+1, len(accounts_list)):
                edges.append((accounts_list[i], accounts_list[j], weight))

    return edges

In [ ]:
# Sniffer C – Semantic Similarity (Cross-Lingual)
def sniffer_semantic(
    posts_dict: Dict[str, dict],
    embeddings_dict: Dict[str, np.ndarray],
    time_buckets: Dict[int, List[str]],
    similarity_threshold: float = 0.85
) -> List[Tuple[str, str, float]]:
    """
    Sniffer C: Link accounts whose posts have high cosine similarity,
    but only if they occur within the same time bucket.

    This is our fix for the "Lone Wolf" Hindi post that didn't share any
    thread/entity with the German post but was clearly pushing the same
    propaganda. The cross-lingual embeddings will bridge them.

    Strategy:
        1. For each time bucket, collect all post_ids and their embedding vectors.
        2. Compute pairwise cosine similarity (only within the bucket).
        3. For any pair with similarity > threshold, add an edge between their
           respective accounts with weight = similarity * 0.5.

    Args:
        posts_dict: dict[post_id] -> post record.
        embeddings_dict: dict[post_id] -> 1024-dim float32 vector.
        time_buckets: dict from build_temporal_buckets().
        similarity_threshold: minimum cosine similarity to consider linking.

    Returns:
        List of (account_i, account_j, weight) tuples.
    """
    edges = []
    # We need a mapping from post_id to account_id
    post_to_account = {post_id: posts_dict[post_id]['account_id'] for post_id in posts_dict}

    print(f"Running Sniffer C on {len(time_buckets)} time buckets...")

    for bucket_start, post_ids in tqdm(time_buckets.items(), desc="Processing buckets"):
        if len(post_ids) < 2:
            continue

        # Retrieve vectors for this bucket, filtering out posts without embeddings
        valid_post_ids = [pid for pid in post_ids if pid in embeddings_dict]
        if len(valid_post_ids) < 2:
            continue

        # Build matrix of vectors
        vectors = np.array([embeddings_dict[pid] for pid in valid_post_ids])

        # Compute cosine similarity for this small matrix
        # Note: vectors are not unit-norm, but cosine_similarity handles that
        try:
            sim_matrix = cosine_similarity(vectors)
        except MemoryError:
            # If a bucket is huge (unlikely at 11k), we'd fall back to chunking.
            # But for this scale, it's fine.
            print(f"Warning: MemoryError on bucket {bucket_start}, skipping.")
            continue

        # Extract upper triangular pairs (i < j) that exceed threshold
        n = len(valid_post_ids)
        for i in range(n):
            for j in range(i+1, n):
                sim = sim_matrix[i, j]
                if sim >= similarity_threshold:
                    acc_i = post_to_account[valid_post_ids[i]]
                    acc_j = post_to_account[valid_post_ids[j]]
                    # Only connect if they are different accounts
                    if acc_i != acc_j:
                        weight = sim * 0.5  # scale semantic weight
                        edges.append((acc_i, acc_j, weight))

    return edges

In [ ]:
# Hypergraph Assembly & Community Detection
def build_weighted_graph(edge_lists: List[List[Tuple[str, str, float]]]) -> nx.Graph:
    """
    Merge multiple edge lists into a single weighted NetworkX graph.

    If the same pair of accounts appears in multiple edge lists (e.g., both
    Sniffer A and Sniffer B caught them), their weights are summed.

    Args:
        edge_lists: A list of edge lists from different sniffers.

    Returns:
        networkx.Graph with weighted edges.
    """
    G = nx.Graph()

    # Accumulate weights in a dictionary
    weight_dict = defaultdict(float)

    for edge_list in edge_lists:
        for u, v, w in edge_list:
            # Ensure consistent ordering to avoid (u,v) vs (v,u) duplicates
            if u < v:
                key = (u, v)
            else:
                key = (v, u)
            weight_dict[key] += w

    # Add edges to the graph
    for (u, v), w in weight_dict.items():
        if w > 0.01:  # ignore negligible weights
            G.add_edge(u, v, weight=w)

    return G

In [ ]:
def extract_candidate_clusters(
    G: nx.Graph,
    posts_dict: Dict[str, dict],
    min_cluster_size: int = 3
) -> List[Dict[str, Union[List[str], List[str], float]]]:
    """
    Run Louvain community detection on the weighted graph, then extract
    candidate clusters that meet the minimum size.

    Each cluster is returned as a dict containing:
        - "post_ids": list of post IDs in the cluster
        - "account_ids": list of unique account IDs
        - "raw_score": average edge weight inside the cluster (to be refined later)

    Args:
        G: Weighted NetworkX graph.
        posts_dict: dict[post_id] -> post record (needed to map accounts to posts).
        min_cluster_size: Minimum number of posts (not accounts) to consider.

    Returns:
        List of candidate cluster dicts.
    """
    if G.number_of_nodes() < 2:
        print("Graph has fewer than 2 nodes. No clusters found.")
        return []

    # Run Louvain
    print(f"Running Louvain on graph with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges...")
    communities = louvain_communities(G, weight='weight', seed=42)

    # Build a reverse map: account_id -> list of post_ids
    account_to_posts = defaultdict(list)
    for post_id, post in posts_dict.items():
        account_to_posts[post['account_id']].append(post_id)

    candidates = []

    for comm in communities:
        # Convert community (set of accounts) to list of posts
        comm_accounts = list(comm)
        comm_posts = []
        for acc in comm_accounts:
            comm_posts.extend(account_to_posts.get(acc, []))

        # Apply minimum size filter (in terms of posts)
        if len(comm_posts) < min_cluster_size:
            continue

        # Compute average edge weight within this community as an initial raw_score
        # (this will be refined in Cell 4 with temporal & profile features)
        internal_edges = []
        for i in range(len(comm_accounts)):
            for j in range(i+1, len(comm_accounts)):
                if G.has_edge(comm_accounts[i], comm_accounts[j]):
                    internal_edges.append(G[comm_accounts[i]][comm_accounts[j]]['weight'])
        avg_weight = np.mean(internal_edges) if internal_edges else 0.0

        candidates.append({
            "account_ids": comm_accounts,
            "post_ids": comm_posts,
            "raw_score": avg_weight
        })

    # Sort by raw_score descending (higher is more promising)
    candidates.sort(key=lambda x: x["raw_score"], reverse=True)

    print(f"Found {len(candidates)} candidate clusters (min posts = {min_cluster_size}).")
    return candidates

In [ ]:
# Main Execution


if 'DATA' not in globals():
    raise RuntimeError("DATA not found! Check if you have run cell above.")

posts_dict = DATA["posts_dict"]
posts_list = DATA['posts_list']
embeddings_dict = DATA['embeddings_dict']
profile_priors = DATA['profile_priors']

# Configuration (borrow from Config)
config = Config
BUCKET_SEC = config.BUCKET_SEC          # 300 seconds (5 min) for time buckets
MAX_WINDOW = config.MAX_TIME_WINDOW_SEC # 1800 seconds (30 min)
SIM_THRESHOLD = config.SEMANTIC_SIM_THRESHOLD
MIN_CLUSTER_SIZE = config.MIN_CLUSTER_SIZE

print("\n" + "="*60)
print("PHASE 2: RUNNING THE 3 SNIFFERS")
print("="*60)

# Step 1: Temporal buckets (for Sniffer C)
print("Building temporal buckets...")
time_buckets = build_temporal_buckets(posts_list, BUCKET_SEC)
print(f"Created {len(time_buckets)} time buckets.")

# Step 2: Run Sniffer A (Structure)
print("\n🔍 Running Sniffer A (Structural)...")
edges_a = sniffer_structure(posts_dict)
print(f"Sniffer A produced {len(edges_a)} edges.")

# Step 3: Run Sniffer B (Entity)
print("\n🔍 Running Sniffer B (Entity)...")
edges_b = sniffer_entity(posts_dict, posts_list)
print(f"Sniffer B produced {len(edges_b)} edges.")

# Step 4: Run Sniffer C (Semantic)
print("\n🔍 Running Sniffer C (Semantic)...")
edges_c = sniffer_semantic(
    posts_dict,
    embeddings_dict,
    time_buckets,
    similarity_threshold=SIM_THRESHOLD
)
print(f"Sniffer C produced {len(edges_c)} edges.")

# Step 5: Merge into a single weighted graph
print("\n📊 Building weighted hypergraph...")
G = build_weighted_graph([edges_a, edges_b, edges_c])
print(f"Graph has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

# Step 6: Extract candidate clusters
print("\n🧩 Extracting candidate clusters via Louvain...")
candidate_clusters = extract_candidate_clusters(G, posts_dict, min_cluster_size=MIN_CLUSTER_SIZE)

# Store candidate clusters back into DATA
DATA['candidate_clusters'] = candidate_clusters
DATA['graph'] = G

## Cell 4: Scoring, Temporal Burst Analysis & Null-Model Calibration

INTENT:
1. For each candidate cluster, compute three sub-scores:
    a) profile_score: average profile_prior of all accounts in the cluster.
    b) temporal_score: how tightly packed the posts are in time (uses Gini coefficient       of inter-arrival times, penalizing Poisson-like distributions).
    c) structural_score: reuse the raw_score from Cell 3 (average edge weight).
 2. Combine them into a final coordination_score = 0.3*profile + 0.4*temporal + 0.3*structural.
 3. Build a NULL distribution by shuffling account_ids 1000 times and re-running the
    scoring pipeline (without re-running Louvain — we just re-score the existing
    cluster structures with shuffled accounts to estimate what random scores look like).
 4. Compute the 99.9th percentile of the null distribution.
 5. For each cluster: if coordination_score > null_threshold, is_coordinated = True.
  6. Store results in DATA['scored_clusters'] for final output.

Intent of this cell:
We now have candidate clusters from Cell 3 (groups of accounts that share some signals). But we need to answer two critical questions:

"How strongly coordinated is this cluster?" → We compute a continuous coordination_score (0.0–1.0) by combining three independent metrics:

Profile Prior Aggregator: Are the accounts in this cluster unusually suspicious (burners, empty profiles, low followers/high posts)?

Temporal Burstiness: Did these posts happen in a tight time window, or are they spread out like an organic conversation?

Structural Density: How many of the 3 sniffers caught them, and with what total edge weight?

"Is this genuinely coordinated or just noise?" → We run a Null Model Permutation Test (shuffle account IDs across posts) to find out what scores random chance produces. If a cluster's score exceeds the 99.9th percentile of the null distribution, we mark it is_coordinated: True. Otherwise, we mark it False.

In [ ]:
import numpy as np
from datetime import datetime
from collections import defaultdict
from tqdm.notebook import tqdm
import random
import copy
import math

In [ ]:
# Custom Gini Coefficient

def compute_gini(values: list) -> float:
    """
    Compute the Gini coefficient for a list of numeric values using NumPy.

    The Gini coefficient measures inequality. For inter-arrival times:
        - Gini = 0 means all intervals are equal (perfectly regular, suspiciously bot-like).
        - Gini = 1 means one interval dominates (highly bursty, coordinated).

    Formula (sorted version):
        G = (sum_{i=1}^{n} (2i - n - 1) * x_i) / (n * sum_{i=1}^{n} x_i)
    where x_i are sorted in ascending order.

    Args:
        values: List of positive numeric values (e.g., time intervals in seconds).

    Returns:
        float: Gini coefficient between 0.0 and 1.0.

    Raises:
        ValueError: If values are empty or sum to zero.
    """
    if not values:
        return 0.0

    # Convert to numpy array for vectorized operations
    arr = np.array(values, dtype=np.float64)

    # Filter out zeros (they break the formula) - only keep positive intervals
    arr = arr[arr > 0]

    if len(arr) == 0:
        return 0.0

    # Sort ascending
    arr = np.sort(arr)
    n = len(arr)

    # Numerator: sum_{i=1}^{n} (2i - n - 1) * x_i
    # Using 1-based indexing: index = i (1..n)
    indices = np.arange(1, n + 1)
    numerator = np.sum((2 * indices - n - 1) * arr)

    # Denominator: n * sum(x_i)
    denominator = n * np.sum(arr)

    if denominator == 0:
        return 0.0

    gini_coeff = numerator / denominator
    # Due to floating point, the result may be slightly outside [0,1].
    # Clamp it to the valid range.
    return max(0.0, min(1.0, gini_coeff))

In [ ]:
#Temporal Burst Score
def compute_temporal_score(post_ids: List[str], posts_dict: Dict[str, dict]) -> float:
    """
    Compute a temporal burstiness score for a cluster of posts.

    Intuition:
        - Coordinated groups post in tight bursts (short inter-arrival times).
        - Organic crowds have Poisson-like arrivals (more spread out).

    We use the Gini coefficient of inter-arrival times:
        - Gini = 0 means all intervals are equal (perfectly regular, like a bot schedule)
        - Gini = 1 means one interval dominates (very bursty)

    We want to reward burstiness but penalize perfectly regular intervals (which could
    be scheduled RSS feeds). So we return:
        score = (1 - Gini) if Gini < 0.1 (too regular -> penalty)
        score = Gini if 0.1 <= Gini <= 0.9 (desired range)
        score = 0.0 if Gini > 0.9 (extremely bursty, but we suspect it's just two posts)

    Additionally, we penalize clusters with only 2 posts because they can easily be
    random co-occurrence.

    Args:
        post_ids: List of post IDs in the cluster.
        posts_dict: dict[post_id] -> post record.

    Returns:
        float: temporal score between 0.0 and 1.0.
    """
    if len(post_ids) < 2:
        return 0.0

    # Extract timestamps and sort them
    timestamps = []
    for pid in post_ids:
        dt = posts_dict[pid].get('created_at')
        if dt is not None:
            timestamps.append(dt.timestamp())

    if len(timestamps) < 2:
        return 0.0

    timestamps.sort()

    # Compute inter-arrival times (differences between consecutive timestamps)
    intervals = [timestamps[i+1] - timestamps[i] for i in range(len(timestamps)-1)]

    # If all intervals are identical -> Gini = 0 (very regular, suspicious in a different way)
    # If one interval is huge -> Gini approaches 1 (very bursty)
    try:
        gini_coeff = compute_gini(intervals)
    except Exception:
        # Fallback if gini calculation fails
        gini_coeff = 0.5

    # Determine the time span of the cluster
    time_span = timestamps[-1] - timestamps[0]

    # Penalize clusters that span more than 2 hours (they are likely organic threads)
    if time_span > 7200:  # 2 hours in seconds
        return 0.0

    # Now map Gini to a score
    if gini_coeff < 0.1:
        # Too regular: likely a scheduled bot. We return a low score.
        return 0.1
    elif gini_coeff <= 0.9:
        # Desired range: higher Gini = more bursty = more coordinated
        return gini_coeff
    else:
        # Extremely bursty (Gini > 0.9) - usually means only 2-3 posts.
        # We down-weight slightly to avoid over-valuing very small clusters.
        return 0.7

In [ ]:
# Combined Scoring Function
def score_candidate_cluster(
    cluster: Dict[str, Union[List[str], List[str], float]],
    posts_dict: Dict[str, dict],
    profile_priors: Dict[str, float]
) -> Dict[str, Union[float, bool]]:
    """
    Compute the final coordination_score for a single cluster by combining:
        1. Profile Prior Aggregator (avg suspicion of accounts).
        2. Temporal Burst Score.
        3. Structural Density (raw_score from graph).

    Weights are: 0.3 (profile) + 0.4 (temporal) + 0.3 (structural).
    These weights were chosen because temporal alignment is the hardest signal
    for organic crowds to fake, so it gets the highest weight.

    Args:
        cluster: Dict with keys 'account_ids', 'post_ids', 'raw_score'.
        posts_dict: dict[post_id] -> post record.
        profile_priors: dict[account_id] -> prior score.

    Returns:
        A dict with the same keys plus:
            - 'coordination_score': combined score (0.0–1.0)
            - 'profile_aggregate': avg profile prior
            - 'temporal_score': temporal burst score
            - 'structural_score': raw_score from graph
    """
    account_ids = cluster['account_ids']
    post_ids = cluster['post_ids']
    structural_score = cluster.get('raw_score', 0.0)

    # Profile Prior Aggregate
    # Average the profile_prior of all accounts in the cluster.
    # If an account is not in profile_priors (shouldn't happen), we give it 0.5.
    profile_scores = [profile_priors.get(acc, 0.5) for acc in account_ids]
    profile_aggregate = np.mean(profile_scores) if profile_scores else 0.0

    # Temporal Score
    temporal_score = compute_temporal_score(post_ids, posts_dict)

    # Structural Score
    # We cap structural_score at 0.8 to prevent it from dominating.
    structural_score_capped = min(structural_score, 0.8)

    # Combine
    # Weights: 0.3 profile, 0.4 temporal, 0.3 structural
    coordination_score = (
        0.3 * profile_aggregate +
        0.4 * temporal_score +
        0.3 * structural_score_capped
    )

    # Clamp to [0.0, 1.0]
    coordination_score = max(0.0, min(1.0, coordination_score))

    # Return enriched cluster
    return {
        'account_ids': account_ids,
        'post_ids': post_ids,
        'raw_score': structural_score,
        'profile_aggregate': profile_aggregate,
        'temporal_score': temporal_score,
        'coordination_score': coordination_score,
        'is_coordinated': False  # will be set later after null model
    }

In [ ]:
# Null Model Permutation Test
def build_null_distribution(
    candidate_clusters: List[Dict],
    posts_dict: Dict[str, dict],
    profile_priors: Dict[str, float],
    n_permutations: int = 1000
) -> np.ndarray:
    """
    Build a null distribution by shuffling account_ids GLOBALLY.

    FIX: Instead of shuffling within each cluster, we shuffle ALL account IDs
    across the ENTIRE dataset. This destroys the structural density of large
    organic crowds, giving us a much more realistic null threshold.
    """
    all_null_scores = []

    # Get all posts from all clusters (flatten the list)
    all_post_ids = []
    for cluster in candidate_clusters:
        all_post_ids.extend(cluster['post_ids'])

    # Get all account IDs that exist in the dataset
    all_account_ids = list(profile_priors.keys())

    print(f"Building null distribution with {n_permutations} permutations (Global shuffle)...")

    for perm_idx in tqdm(range(n_permutations), desc="Permutations"):
        # Shuffle the ENTIRE list of account IDs globally
        shuffled_accounts_global = all_account_ids.copy()
        random.shuffle(shuffled_accounts_global)

        # Map each post to a shuffled account (cycling through the shuffled list)
        # This ensures we preserve the number of posts per cluster,
        # but the accounts are completely random across the whole dataset.
        shuffled_map = {}
        for idx, pid in enumerate(all_post_ids):
            shuffled_map[pid] = shuffled_accounts_global[idx % len(shuffled_accounts_global)]

        # Now score each cluster using the globally shuffled accounts
        for cluster in candidate_clusters:
            post_ids = cluster['post_ids']
            structural_score = cluster.get('raw_score', 0.0)

            # Get the shuffled account IDs for just this cluster's posts
            shuffled_accounts = [shuffled_map[pid] for pid in post_ids]

            # Compute profile_aggregate for this shuffled assignment
            profile_scores = [profile_priors.get(acc, 0.5) for acc in shuffled_accounts]
            profile_agg = np.mean(profile_scores) if profile_scores else 0.0

            # Temporal score is unchanged (timestamps are fixed to posts)
            if 'temporal_score_cache' not in cluster:
                # We cache it to avoid recomputing for every permutation
                cluster['temporal_score_cache'] = compute_temporal_score(post_ids, posts_dict)
            temporal_score = cluster['temporal_score_cache']

            # Structural score is also capped and reused
            structural_score_capped = min(structural_score, 0.8)

            # Combine using the same weights
            null_score = (
                0.3 * profile_agg +
                0.4 * temporal_score +
                0.3 * structural_score_capped
            )
            all_null_scores.append(null_score)

    return np.array(all_null_scores)

In [ ]:
def calibrate_and_score_clusters(
    candidate_clusters: List[Dict],
    posts_dict: Dict[str, dict],
    profile_priors: Dict[str, float],
    n_permutations: int = 1000,
    percentile_threshold: float = 99.9
) -> List[Dict]:
    """
    Main orchestration function for Cell 4.

    1. Score each candidate cluster.
    2. Build null distribution.
    3. Determine threshold.
    4. Mark is_coordinated based on threshold.
    5. Return the list of scored clusters.

    Args:
        candidate_clusters: Output from Cell 3.
        posts_dict: dict[post_id] -> post record.
        profile_priors: dict[account_id] -> prior.
        n_permutations: Number of permutations for null model.
        percentile_threshold: Percentile to use as cutoff (e.g., 99.9).

    Returns:
        List of scored cluster dicts with 'is_coordinated' and 'coordination_score'.
    """
    # Step 1: Score each cluster
    print("Scoring candidate clusters...")
    scored_clusters = []
    for cluster in tqdm(candidate_clusters, desc="Scoring"):
        scored = score_candidate_cluster(cluster, posts_dict, profile_priors)
        scored_clusters.append(scored)

    # Step 2: Build null distribution
    print("\nRunning null model calibration...")
    null_scores = build_null_distribution(
        candidate_clusters,
        posts_dict,
        profile_priors,
        n_permutations=n_permutations
    )

    # Step 3: Compute threshold
    threshold = np.percentile(null_scores, percentile_threshold)
    print(f"\nNull model: {percentile_threshold}th percentile = {threshold:.4f}")
    print(f"Number of null samples: {len(null_scores)}")

    #Step 4: Apply threshold
    for cluster in scored_clusters:
        if cluster['coordination_score'] > threshold:
            cluster['is_coordinated'] = True
        else:
            cluster['is_coordinated'] = False

    # Sort by coordination_score descending
    scored_clusters.sort(key=lambda x: x['coordination_score'], reverse=True)

    # Print summary
    n_true = sum(1 for c in scored_clusters if c['is_coordinated'])
    n_false = len(scored_clusters) - n_true
    print(f"\n✅ Calibration complete: {n_true} clusters flagged as coordinated, {n_false} as noise.")

    # Print top 5 scores for manual inspection
    print("\n📊 Top 5 clusters by coordination_score:")
    for i, cluster in enumerate(scored_clusters[:5]):
        print(f"  {i+1}. Score: {cluster['coordination_score']:.4f} | "
              f"Posts: {len(cluster['post_ids'])} | "
              f"Accounts: {len(cluster['account_ids'])} | "
              f"Coordinated: {cluster['is_coordinated']}")

    return scored_clusters

In [ ]:
# Main Execution

if 'DATA' not in globals():
    raise RuntimeError("DATA not found! Please run Cell 2 and Cell 3 first.")

candidate_clusters = DATA.get('candidate_clusters', [])
if not candidate_clusters:
    print("⚠️ No candidate clusters found. Skipping scoring.")
    DATA['scored_clusters'] = []
else:
    posts_dict = DATA['posts_dict']
    profile_priors = DATA['profile_priors']

    # Pull configuration from Config
    config = Config
    n_permutations = config.NULL_PERMUTATIONS
    percentile_threshold = config.PERCENTILE_THRESHOLD

    print("\n" + "="*60)
    print("PHASE 4: SCORING & NULL-MODEL CALIBRATION")
    print("="*60)
    print(f"Processing {len(candidate_clusters)} candidate clusters...")
    print(f"Using {n_permutations} permutations for null model.")
    print(f"Threshold: {percentile_threshold}th percentile.")

    # Run the scoring & calibration
    scored_clusters = calibrate_and_score_clusters(
        candidate_clusters,
        posts_dict,
        profile_priors,
        n_permutations=n_permutations,
        percentile_threshold=percentile_threshold
    )

    # Store back into DATA
    DATA['scored_clusters'] = scored_clusters
    DATA['null_threshold'] = np.percentile(
        build_null_distribution(
            candidate_clusters,
            posts_dict,
            profile_priors,
            n_permutations=min(100, n_permutations)  # quick preview for the threshold
        ) if 'null_scores' not in DATA else DATA.get('null_scores', []),
        percentile_threshold
    )

## Cell 5: Final Output – results.json

 INTENT:
 1. Retrieve the scored clusters from DATA['scored_clusters'].
 2. Filter to only those where is_coordinated is True.
 3. Sort them by coordination_score descending (highest confidence first).
 4. Transform each cluster into the required JSON schema:#    {"post_ids": [...], "is_coordinated": true, "coordination_score": float}
 5. Write the JSON file to the appropriate tier folder (dev/ or eval/).
 6. Print a detailed summary for manual inspection.

In [ ]:
import json
import os
from typing import List, Dict, Any

def prepare_results(
    scored_clusters: List[Dict[str, Any]],
    tier: str = "dev"
) -> Dict[str, List[Dict[str, Any]]]:
    """
    Filter scored clusters and format them for the results.json schema.

    We only keep clusters that were flagged as coordinated (is_coordinated == True).
    We sort them by coordination_score descending so the most confident groups appear first.

    Args:
        scored_clusters: List of scored cluster dicts from Cell 4.
        tier: "dev" or "eval" (used only for logging).

    Returns:
        A dict with key "clusters" containing the filtered and formatted list.

    Example output:
        {
            "clusters": [
                {
                    "post_ids": ["p_abc", "p_def"],
                    "is_coordinated": true,
                    "coordination_score": 0.92
                }
            ]
        }
    """
    # Filter to only coordinated clusters
    coordinated = [c for c in scored_clusters if c.get('is_coordinated', False)]

    # Sort by score descending
    coordinated.sort(key=lambda x: x.get('coordination_score', 0.0), reverse=True)

    # Format according to schema
    formatted_clusters = []
    for cluster in coordinated:
        formatted_clusters.append({
            "post_ids": cluster.get('post_ids', []),
            "is_coordinated": True,
            "coordination_score": cluster.get('coordination_score', 0.0)
        })

    return {"clusters": formatted_clusters}

In [ ]:
# Save to File
def save_results_json(
    results: Dict[str, List[Dict[str, Any]]],
    tier: str = "dev",
    output_filename: str = "results.json"
) -> str:
    """
    Write the results dictionary to a JSON file in the appropriate tier folder.

    Args:
        results: The formatted results dict from prepare_results().
        tier: "dev" or "eval".
        output_filename: Name of the output file (default: "results.json").

    Returns:
        str: The full path to the saved file.

    Raises:
        FileNotFoundError: If the tier folder does not exist.
    """
    # Get the base path for the tier
    try:
        base_path = Config.get_data_path(tier)
    except NameError:
        # Fallback if Config isn't available
        base_path = f"/content/data/{tier}/"
        print(f"⚠️ Config not found. Using default path: {base_path}")

    # Ensure the directory exists
    os.makedirs(base_path, exist_ok=True)

    # Full output path
    output_path = os.path.join(base_path, output_filename)

    # Write the JSON file with pretty formatting (indent=2) for readability
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    return output_path

In [ ]:
#Summary Report for Manual Inspection
def print_summary_report(
    results: Dict[str, List[Dict[str, Any]]],
    tier: str = "dev"
) -> None:
    """
    Print a clear summary of the output to help with manual inspection.

    This includes:
        - Total number of coordinated clusters found.
        - Distribution of coordination scores.
        - Detailed listing of the top 10 clusters with their post counts
          and sample post IDs (so the user can quickly verify them).

    Args:
        results: The formatted results dict.
        tier: "dev" or "eval" (used for logging).
    """
    clusters = results.get('clusters', [])
    n_clusters = len(clusters)

    print("\n" + "="*70)
    print(f"📄 FINAL RESULTS SUMMARY (TIER: {tier.upper()})")
    print("="*70)
    print(f"✅ Total coordinated clusters identified: {n_clusters}")

    if n_clusters == 0:
        print("⚠️ No clusters were flagged as coordinated.")
        print("   This may happen if the null threshold is too strict.")
        print("   Try lowering Config.PERCENTILE_THRESHOLD or check the data.")
        return

    # Compute score distribution
    scores = [c['coordination_score'] for c in clusters]
    print(f"\n📊 Coordination Score Distribution:")
    print(f"   Min:    {min(scores):.4f}")
    print(f"   Max:    {max(scores):.4f}")
    print(f"   Mean:   {np.mean(scores):.4f}")
    print(f"   Median: {np.median(scores):.4f}")

    # Print top 10 clusters for manual inspection
    print("\n🔍 TOP 10 COORDINATED CLUSTERS (for manual verification):")
    print("-" * 70)

    for idx, cluster in enumerate(clusters[:10], start=1):
        post_ids = cluster['post_ids']
        score = cluster['coordination_score']
        n_posts = len(post_ids)
        n_accounts = len(set(post_ids))  # approximate, but we could map to accounts

        # Show a sample of up to 5 post IDs
        sample_ids = post_ids[:5]
        sample_str = ", ".join(sample_ids)
        if len(post_ids) > 5:
            sample_str += f" ... and {len(post_ids) - 5} more"

        print(f"\n  #{idx}: Score = {score:.4f}")
        print(f"      Posts: {n_posts} | Sample IDs: {sample_str}")

    print("\n" + "="*70)
    print("✅ Manual inspection tip: Compare these clusters against the raw")
    print("   posts to confirm they represent genuine coordination (shared")
    print("   rare URLs, exact thread hijacking, cross-lingual semantic overlap).")
    print("="*70 + "\n")

In [ ]:
#Main Execution


if 'DATA' not in globals():
    raise RuntimeError("DATA not found! Please run Cell 2, Cell 3, and Cell 4 first.")

scored_clusters = DATA.get('scored_clusters', [])
if not scored_clusters:
    print("⚠️ No scored clusters found. Run Cell 4 first.")
    DATA['results'] = {"clusters": []}
else:
    # Determine which tier we are working on (default to "dev")
    tier = DATA.get('tier', 'dev')
    print(f"\n📁 Generating results for tier: {tier.upper()}")

    # Step 1: Prepare results
    print("\n📝 Formatting results...")
    results = prepare_results(scored_clusters, tier=tier)

    # Step 2: Print summary report for manual inspection
    print_summary_report(results, tier=tier)

    # Step 3: Save to file
    output_path = save_results_json(results, tier=tier, output_filename="results.json")
    print(f"💾 Results saved to: {output_path}")

    # Step 4: Store in DATA for reference
    DATA['results'] = results
    DATA['results_path'] = output_path

print("\n✅ Cell 5 complete.")
print("🎯 You can now download the results.json file from the path shown above.")
print("   For eval tier, this is your final submission artifact.")


In [ ]:
# Reload the file to ensure it's valid JSON and print a preview
try:
    with open(output_path, 'r', encoding='utf-8') as f:
        loaded = json.load(f)
        print("\n🔎 Validation: File loaded successfully.")
        print(f"   Contains {len(loaded.get('clusters', []))} coordinated clusters.")
except Exception as e:
    print(f"\n❌ Validation failed: {e}")

## Cell 6: Visualization Suite

# INTENT:
1. Plot the weighted hypergraph, highlighting the top coordinated clusters.
2. Plot a histogram of coordination scores with the null threshold overlay.
3. Plot a rug/timeline comparison (top cluster vs. organic baseline).
4. Plot a scatter of cluster size vs. coordination score, colored by is_coordinated.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import networkx as nx
from datetime import datetime
import random

# Set a clean, professional plotting style
sns.set_style("whitegrid")
sns.set_palette("Set2")
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 12


#Data Preparation for Visualizations


if 'DATA' not in globals():
    raise RuntimeError("DATA not found! Please run Cells 1-5 first.")

scored_clusters = DATA.get('scored_clusters', [])
graph = DATA.get('graph', nx.Graph())
profile_priors = DATA.get('profile_priors', {})
posts_dict = DATA.get('posts_dict', {})
tier = DATA.get('tier', 'dev')

# We'll need the null threshold. If not stored, we compute it roughly.
# In practice, it was printed in Cell 4. We'll estimate it from the scored clusters.
null_threshold = 0.0
if scored_clusters:
    # The threshold is the 99.9th percentile. We can approximate it from the scores.
    # Alternatively, we can just use a fixed visual line at 0.55 (typical).
    # Let's try to recover it from the data: if all coord clusters have score > X,
    # and non-coord have score < X, we find the boundary.
    coord_scores = [c['coordination_score'] for c in scored_clusters if c.get('is_coordinated', False)]
    non_coord_scores = [c['coordination_score'] for c in scored_clusters if not c.get('is_coordinated', False)]

    if coord_scores and non_coord_scores:
        # A simple boundary: average of min(coord) and max(non_coord)
        null_threshold = (min(coord_scores) + max(non_coord_scores)) / 2
    elif coord_scores:
        null_threshold = min(coord_scores) - 0.05
    else:
        null_threshold = 0.6  # fallback

print(f"Using visual threshold: {null_threshold:.4f}")

In [ ]:
#Visualization 1 – The Weighted Hypergraph (Top Clusters)

def plot_hypergraph(graph, scored_clusters, profile_priors, top_k=3):
    """Plot the subgraph containing the top K coordinated clusters."""
    print("\n📊 Plotting Hypergraph...")

    # Find the top K coordinated clusters
    coord_clusters = [c for c in scored_clusters if c.get('is_coordinated', False)]
    coord_clusters.sort(key=lambda x: x['coordination_score'], reverse=True)
    top_clusters = coord_clusters[:top_k]

    if not top_clusters:
        print("   No coordinated clusters to plot.")
        return

    # Collect all account IDs from the top clusters
    nodes_to_keep = set()
    for cluster in top_clusters:
        nodes_to_keep.update(cluster['account_ids'])

    # Extract the subgraph
    subgraph = graph.subgraph(nodes_to_keep).copy()

    if subgraph.number_of_nodes() < 2:
        print("   Subgraph too small to visualize.")
        return

    # Compute node colors based on profile prior
    node_colors = []
    for node in subgraph.nodes():
        prior = profile_priors.get(node, 0.5)
        # Red (high suspicion) to Blue (low suspicion)
        node_colors.append((prior, 0.2, 1 - prior))  # RGB: R=prior, B=1-prior

    # Edge widths based on weight
    edge_weights = [subgraph[u][v]['weight'] for u, v in subgraph.edges()]
    max_weight = max(edge_weights) if edge_weights else 1.0
    edge_widths = [w / max_weight * 3 + 0.5 for w in edge_weights]

    # Compute layout (spring layout with seed for reproducibility)
    pos = nx.spring_layout(subgraph, seed=42, k=0.5)

    # Create the figure
    plt.figure(figsize=(12, 10))

    # Draw the graph
    nx.draw_networkx_nodes(subgraph, pos, node_size=200, node_color=node_colors, alpha=0.9)
    nx.draw_networkx_edges(subgraph, pos, width=edge_widths, alpha=0.6, edge_color='gray')
    nx.draw_networkx_labels(subgraph, pos, font_size=8, font_weight='bold')

    # Add a title and legend
    plt.title(f"Hypergraph of Top {top_k} Coordinated Clusters (Tier: {tier})\n"
              f"Node Color: Red = Suspicious Profile | Blue = Organic Profile\n"
              f"Edge Width: Strength of Coordination Signal",
              fontsize=14)

    # Create custom legend patches
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='red', alpha=0.7, label='High Suspicion (Profile Prior > 0.7)'),
        Patch(facecolor='blue', alpha=0.7, label='Low Suspicion (Profile Prior < 0.3)'),
        Patch(facecolor='purple', alpha=0.7, label='Mixed / Unknown (Profile Prior ~ 0.5)')
    ]
    plt.legend(handles=legend_elements, loc='upper right')

    plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# Visualization 2 – Score Distribution & Null Threshold


def plot_score_distribution(scored_clusters, null_threshold):
    """Plot a histogram of coordination scores with the null threshold overlay."""
    print("\n📊 Plotting Score Distribution...")

    if not scored_clusters:
        print("   No clusters to plot.")
        return

    scores = [c['coordination_score'] for c in scored_clusters]
    coord_flags = [c.get('is_coordinated', False) for c in scored_clusters]

    # Separate coordinated vs. non-coordinated
    coord_scores = [s for s, f in zip(scores, coord_flags) if f]
    non_coord_scores = [s for s, f in zip(scores, coord_flags) if not f]

    plt.figure(figsize=(12, 6))

    # Plot histograms with transparency
    plt.hist(non_coord_scores, bins=20, alpha=0.6, label='Not Coordinated (Noise)', color='blue', edgecolor='black')
    plt.hist(coord_scores, bins=20, alpha=0.8, label='Coordinated (Signal)', color='red', edgecolor='black')

    # Add vertical line for the null threshold
    plt.axvline(x=null_threshold, color='green', linestyle='--', linewidth=2,
                label=f'Null Threshold ({null_threshold:.3f})')

    # Annotate the threshold
    plt.text(null_threshold + 0.01, plt.ylim()[1] * 0.9,
             f'Threshold\n{null_threshold:.3f}',
             rotation=0, fontsize=12, color='green', ha='left')

    plt.xlabel('Coordination Score')
    plt.ylabel('Number of Clusters')
    plt.title(f'Score Distribution vs. Null Threshold (Tier: {tier})\n'
              f'Signal: {len(coord_scores)} clusters | Noise: {len(non_coord_scores)} clusters',
              fontsize=14)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
# Visualization 3 – Temporal Burst Comparison (Top Cluster vs Organic Baseline)

def plot_temporal_burst(scored_clusters, posts_dict, null_threshold):
    """Plot a rug plot comparing the top coordinated cluster's timestamps vs organic baseline."""
    print("\n📊 Plotting Temporal Burst Comparison...")

    # Get the top coordinated cluster
    coord_clusters = [c for c in scored_clusters if c.get('is_coordinated', False)]
    if not coord_clusters:
        print("   No coordinated clusters to plot.")
        return

    top_cluster = max(coord_clusters, key=lambda x: x['coordination_score'])
    top_post_ids = top_cluster['post_ids']

    # Get timestamps of the top cluster
    top_timestamps = []
    for pid in top_post_ids:
        dt = posts_dict[pid].get('created_at')
        if dt is not None:
            top_timestamps.append(dt.timestamp())

    if len(top_timestamps) < 2:
        print("   Top cluster has insufficient timestamps.")
        return

    # Get a random sample of organic posts (non-coordinated clusters)
    non_coord_clusters = [c for c in scored_clusters if not c.get('is_coordinated', False)]
    random_clusters = random.sample(non_coord_clusters, min(3, len(non_coord_clusters)))

    baseline_timestamps = []
    for cluster in random_clusters:
        for pid in cluster['post_ids'][:10]:  # Take up to 10 per cluster to balance
            dt = posts_dict[pid].get('created_at')
            if dt is not None:
                baseline_timestamps.append(dt.timestamp())

    if not baseline_timestamps:
        # Fallback: take any 20 random posts
        all_post_ids = list(posts_dict.keys())
        sample_ids = random.sample(all_post_ids, min(20, len(all_post_ids)))
        for pid in sample_ids:
            dt = posts_dict[pid].get('created_at')
            if dt is not None:
                baseline_timestamps.append(dt.timestamp())

    # Normalize timestamps to minutes for readability (relative to the first timestamp)
    ref_time = min(top_timestamps + baseline_timestamps)
    top_minutes = [(t - ref_time) / 60 for t in top_timestamps]
    baseline_minutes = [(t - ref_time) / 60 for t in baseline_timestamps]

    # Create the plot
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

    # Top cluster (Rug plot with vertical lines)
    ax1.scatter(top_minutes, [1] * len(top_minutes), marker='|', s=200, color='red', alpha=0.8)
    ax1.set_title(f'Top Coordinated Cluster (Score: {top_cluster["coordination_score"]:.4f})', fontsize=12)
    ax1.set_ylabel('Posts')
    ax1.set_ylim(0.8, 1.2)
    ax1.set_yticks([])

    # Baseline organic posts
    ax2.scatter(baseline_minutes, [1] * len(baseline_minutes), marker='|', s=200, color='blue', alpha=0.5)
    ax2.set_title('Baseline Organic Posts (Random Sample)', fontsize=12)
    ax2.set_xlabel('Time (minutes relative to first post)')
    ax2.set_ylabel('Posts')
    ax2.set_ylim(0.8, 1.2)
    ax2.set_yticks([])

    plt.suptitle(f'Temporal Burst Comparison (Tier: {tier})', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
#Visualization 4 – Cluster Size vs. Coordination Score

def plot_size_vs_score(scored_clusters, null_threshold):
    """Scatter plot: number of posts per cluster vs coordination_score, colored by is_coordinated."""
    print("\n📊 Plotting Cluster Size vs. Score...")

    if not scored_clusters:
        print("   No clusters to plot.")
        return

    sizes = [len(c['post_ids']) for c in scored_clusters]
    scores = [c['coordination_score'] for c in scored_clusters]
    colors = ['red' if c.get('is_coordinated', False) else 'blue' for c in scored_clusters]

    plt.figure(figsize=(12, 7))

    # Scatter plot
    plt.scatter(scores, sizes, c=colors, alpha=0.7, s=80, edgecolors='black', linewidth=0.5)

    # Add the null threshold vertical line
    plt.axvline(x=null_threshold, color='green', linestyle='--', linewidth=2,
                label=f'Null Threshold ({null_threshold:.3f})')

    plt.xlabel('Coordination Score', fontsize=12)
    plt.ylabel('Number of Posts in Cluster', fontsize=12)
    plt.title(f'Cluster Size vs. Coordination Score (Tier: {tier})\n'
              'Red = Flagged Coordinated | Blue = Marked Noise',
              fontsize=14)
    plt.legend()
    plt.grid(alpha=0.3)

    # Annotate the top cluster
    if scored_clusters:
        top = max(scored_clusters, key=lambda x: x['coordination_score'])
        plt.annotate(f"Top Cluster\nScore: {top['coordination_score']:.3f}",
                     xy=(top['coordination_score'], len(top['post_ids'])),
                     xytext=(top['coordination_score'] + 0.05, len(top['post_ids']) + 2),
                     arrowprops=dict(facecolor='black', shrink=0.05),
                     fontsize=10)

    plt.tight_layout()
    plt.show()

In [ ]:
# Main Execution for Cell 6 – Run All Visualizations

print("\n" + "="*70)
print("📊 VISUALIZATION SUITE (Tier: {})".format(tier.upper()))
print("="*70)

# Run all visualizations
plot_hypergraph(graph, scored_clusters, profile_priors, top_k=10)
plot_score_distribution(scored_clusters, null_threshold)
plot_temporal_burst(scored_clusters, posts_dict, null_threshold)
plot_size_vs_score(scored_clusters, null_threshold)

print("\n" + "="*70)
print("✅ Visualizations complete.")
print("💡 Use these plots to validate your manual inspection:")
print("   - Red nodes in the graph should represent suspicious, tightly-connected accounts.")
print("   - The score distribution should show a clear separation between signal and noise.")
print("   - The temporal burst plot should show the coordinated cluster firing in a tight window.")
print("   - The size-vs-score plot should show small, dense clusters scoring high.")
print("="*70)

# Save the figures to the tier folder (optional)
# If you want to save them, uncomment the lines below:
# output_dir = f"/content/data/{tier}/figures/"
# os.makedirs(output_dir, exist_ok=True)
# plt.savefig(os.path.join(output_dir, "score_distribution.png"), dpi=150, bbox_inches='tight')

In [ ]:
# Run this in a new Colab cell to inspect the top 10 candidates
import json

# Load the scored clusters from Cell 4 (if you haven't saved them, re-run Cell 4)
scored_clusters = DATA.get('scored_clusters', [])

if not scored_clusters:
    print("No scored clusters found. Run Cell 4 first.")
else:
    print("="*70)
    print("TOP 10 CANDIDATE CLUSTERS (Including those marked 'False')")
    print("="*70)

    for idx, cluster in enumerate(scored_clusters[:10], start=1):
        print(f"\n#{idx}: Score = {cluster['coordination_score']:.4f} | Flagged: {cluster['is_coordinated']}")
        print(f"   Posts: {len(cluster['post_ids'])} | Accounts: {len(cluster['account_ids'])}")

        # Show a sample of post IDs so you can manually inspect them
        sample = cluster['post_ids'][:3]
        print(f"   Sample Post IDs: {', '.join(sample)}")

        # Show profile prior aggregate (avg suspicion of accounts in this cluster)
        print(f"   Avg Profile Prior: {cluster.get('profile_aggregate', 0.0):.3f}")
        print(f"   Temporal Score: {cluster.get('temporal_score', 0.0):.3f}")
        print("-"*50)

In [ ]:
#code to if the two clusters caught are pushing coordinated content
import json

# Load results
with open('/content/data/eval/results.json', 'r') as f:
    results = json.load(f)

# Load posts
posts = []
with open('/content/data/eval/posts.jsonl', 'r') as f:
    for line in f:
        posts.append(json.loads(line))

for idx, cluster in enumerate(results['clusters'], start=1):
    print(f"\n{'='*60}")
    print(f"CLUSTER #{idx} (Score: {cluster['coordination_score']:.4f})")
    print(f"{'='*60}")
    for pid in cluster['post_ids']:
        for post in posts:
            if post['post_id'] == pid:
                print(f"  Post: {post['post_id']}")
                print(f"  Account: {post['account_id']}")
                print(f"  Time: {post['created_at']}")
                print(f"  Text (first 200 chars): {post['text'][:200]}...")
                print(f"  Thread: {post['thread_id']}")
                print(f"  URLs: {post['urls']}")
                print("-"*40)
                break